# GPA778 - Laboratoire 2
# Partie B - Perception II : Représentation 3D par balayages Lidar 2D

Lors du dernier laboratoire (2a), vous avez créé le package `jerro_perception`, avec un launch file et deux nodes (`cam_traitement` et `lidar_traitement`). Ces fichiers ont permis de démarrer l'acquisition des données de perception, mais encore peu de travail a été effectué en ce qui a trait aux traitements de ces données.

De plus, pour le moment, l'ordinateur de bord du robot (le RPI) n'a fait que l'acquisition de données de perception 2D. Bien qu'il soit possible de faire de la navigation autonome à partir de données de perception 2D, nous voudrons maintenant obtenir une perception plus complète de l'environnement grâce à une perception 3D.

**Objectif :** Reconstruire l'environnement à proximité du robot en 3D, grâce à l'acquisition d'une succession de balayages lidar 2D.

## 2.2.1 Objectifs académiques

L'objectif de ce laboratoire est d'arriver à faire l'acquisition d'une succession de balayages lidar 2D, afin de créer une représentation 3D de l'environnement à proximité du robot. Pour ce faire, il faudra bien sûr faire pivoter le plan d'acquisition du lidar, en se servant du **servomoteur** permettant d'accomplir cette tâche.

### Objectifs spécifiques :

- Développer le modèle de projection des données pseudo-sphériques en données cartésiennes, exprimées dans le repère du robot (`base_link`), et le **vérifier** avec la chaîne TF publiée à partir de l'URDF
- Se familiariser avec le concept d'**"action"** sous ROS, en ajoutant une action au node `lidar_traitement.py` et en codant un callback associé
- Commander le servomoteur par l'intermédiaire d'un **service** ROS (`/set_servo_pos`)
- Utiliser la classe **`ScanAssembler`** (package `laser_assembler`) pour assembler une succession de messages `LaserScan` en un `PointCloud2`, à l'aide des transformations TF
- Approfondir la notion de TF et l'utilisation du package [joint_state_publisher](https://github.com/ros/joint_state_publisher/blob/ros2/joint_state_publisher/README.md)

## 2.2.2 Jalon 1 : Développement du modèle de projection des données pseudo-sphériques à cartésiennes dans le repère du robot

### Problématique

Puisque l'axe de rotation faisant pivoter le lidar ne passe pas par le centre optique de ce dernier, on se retrouve avec des **coordonnées pseudo-sphériques**, nous empêchant ainsi d'utiliser les formules de projection bien connues.

Il faut donc développer les équations permettant d'exprimer les points du lidar dans le **repère du robot (`base_link`)**.

Les référentiels du robot sont décrits dans le fichier URDF du package `jerro_description` (`urdf/jerro.urdf`). Comme vu au laboratoire 1c, `robot_state_publisher` calcule à partir de cet URDF les transformations entre les référentiels et les publie dans TF. Dans ce jalon, vous développez le modèle à la main ; vous le comparerez ensuite à ce que publie TF.

### Identification de la matrice de transformation R_T_L

On cherche à identifier la matrice de transformation homogène $^RT_L$, où :
- **R** fait référence au repère du robot
- **L** fait référence au repère du lidar (origine au centre optique)

![Figure 2.4 - Repère du Lidar](figure_2_4.png)
*Figure 2.4: Repère du lidar, tel que spécifié par le fabricant. À gauche : vue du dessus et à droite : vue de face.*

![Figure 2.5 - Définition des repères](figure_2_5.png)
*Figure 2.5: Définition des repères pertinents, qui permettront d'identifier R_T_L*

### Repères définis

Sur l'image, plusieurs repères sont définis :
- **Repère L** : Repère du lidar (tel qu'indiqué par le fabricant)
- **Repère R** : Repère du robot
- **Repère S** : Repère additionnel avec origine à l'intersection de l'axe de rotation du servomoteur et le plan $x_R - z_R$

![Figure 2.6 - Relations géométriques](figure_2_6.png)
*Figure 2.6: Identification des repères pertinents ainsi que leurs relations géométriques.*

**Note importante :** Le lidar est pivoté d'un angle $\psi$ par rapport au servomoteur. Lorsque $\psi = 0$, l'axe $z_L$ est colinéaire avec l'axe $x_S$ et pointe dans la direction inverse de ce dernier.

En pratique, nous ne permettrons pas au lidar de pivoter à de si petits angles, car celui-ci entrerait alors en collision avec le boîtier du RPI.

### Équations de transformation homogène

Pour passer du repère R au repère L, on utilise des transformations homogènes :

$$^RT_L = Trans_{xyz}(0.04, 0, 0.034) \cdot Rot_y(\psi - \pi/2) \cdot Trans_{xyz}(0, 0, 0.113)$$

De façon plus explicite :

$$^RT_L = \begin{bmatrix}
1 & 0 & 0 & 0.04 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0.034 \\
0 & 0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
\cos(\psi - \pi/2) & 0 & \sin(\psi - \pi/2) & 0 \\
0 & 1 & 0 & 0 \\
-\sin(\psi - \pi/2) & 0 & \cos(\psi - \pi/2) & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0.113 \\
0 & 0 & 0 & 1
\end{bmatrix}$$

### Résultat final

En effectuant les multiplications matricielles et en simplifiant, on obtient :

$$^RT_L = \begin{bmatrix}
\cos(\psi - \pi/2) & 0 & \sin(\psi - \pi/2) & 0.04 + 0.113 \cdot \sin(\psi - \pi/2) \\
0 & 1 & 0 & 0 \\
-\sin(\psi - \pi/2) & 0 & \cos(\psi - \pi/2) & 0.034 + 0.113 \cdot \cos(\psi - \pi/2) \\
0 & 0 & 0 & 1
\end{bmatrix}$$

Forme compacte :

$$^RT_L = \begin{bmatrix}
Rot(y, \psi - \pi/2)_{3×3} & P_{3×1} \\
0_{1×3} & 1
\end{bmatrix}$$

Où :

$$R(y, \psi - \pi/2) = \begin{bmatrix}
\cos(\psi - \pi/2) & 0 & \sin(\psi - \pi/2) \\
0 & 1 & 0 \\
-\sin(\psi - \pi/2) & 0 & \cos(\psi - \pi/2)
\end{bmatrix}$$

$$P = \begin{bmatrix}
0.04 + 0.113 \cdot \sin(\psi - \pi/2) \\
0 \\
0.034 + 0.113 \cdot \cos(\psi - \pi/2)
\end{bmatrix}$$

### Vérification avec TF

Dans le robot, la transformation $^RT_L$ n'est pas calculée par votre code : elle est publiée automatiquement dans TF par la chaîne suivante :

```mermaid
flowchart LR
    SRV["service<br/>/set_servo_pos"] --> SV[servomoteur]
    SV -- "/set_joint_states<br/>(JointState)" --> JSP[joint_state_publisher]
    JSP --> RSP["robot_state_publisher<br/>(URDF)"]
    RSP -- "TF" --> TF(["base_link → servomotor → lidar"])
```

Le node `servomoteur` convertit la largeur d'impulsion commandée (en µs) en angle du joint `base_link_to_servomotor` :

$$\theta = 0.623334 + (2000 - \text{pulse}_{\mu s}) \times (-0.001558335) \quad \text{[rad]}$$

**À faire :** une fois le robot démarré (voir Jalon 3), placez le servo à quelques positions et comparez la transformation publiée avec celle obtenue par vos équations :

```bash
ros2 service call /set_servo_pos jerro_msgs/srv/SetServoPos "{position: 1500}"
ros2 run tf2_ros tf2_echo base_link lidar
```

Quel lien faites-vous entre l'angle $\theta$ du joint et l'angle $\psi$ de vos équations ? Les dimensions de l'URDF correspondent-elles à celles de la figure 2.6 ?

## 2.2.3 Jalon 2 : Qu'est-ce qu'un serveur d'action ?

Pour reconstruire l'environnement en 3D, le robot doit enchaîner plusieurs secondes de travail : déplacer le servo, prendre un scan, recommencer… On veut pouvoir **lancer ce balayage depuis un autre node** (ou la ligne de commande) et **suivre sa progression**. C'est le rôle d'une **action**.

Vous connaissez déjà deux façons de communiquer entre nodes. L'action est la troisième :

| | Topic | Service | Action |
|---|---|---|---|
| Échange | flux continu, sans réponse | 1 requête → 1 réponse | 1 objectif → des nouvelles… → 1 résultat |
| Durée | — | courte (quelques ms) | longue (secondes, minutes) |
| Dans notre robot | `/scan` (le lidar publie en continu) | `/set_servo_pos` (bouger le servo) | `/perception/start_scan` (balayage 3D) |

> Une action, c'est un service **long**, qui **donne des nouvelles** pendant qu'il travaille.

Ce jalon est une lecture : vous écrirez le code au Jalon 3.

### Les trois messages d'une action

Une action est définie dans un fichier `.action`, en trois parties séparées par `---`. Voici celle de notre balayage (`jerro_msgs/action/StartScan.action`) :

```
# Objectif (goal) : ce que le client demande
bool start
---
# Résultat (result) : envoyé une seule fois, à la fin
sensor_msgs/PointCloud2 point_cloud
bool success
string message
---
# Retour (feedback) : envoyé plusieurs fois, pendant le travail
float32 progress
```

Pour l'afficher sur le robot :
```bash
ros2 interface show jerro_msgs/action/StartScan
```

### Déroulement d'une action

Le **client** (ici, la ligne de commande) envoie un objectif. Le **serveur** (ici, votre node `lidar_traitement`) l'accepte, envoie des feedbacks pendant le travail, puis le résultat :

```mermaid
sequenceDiagram
    participant C as Client<br/>(ros2 action send_goal)
    participant S as Serveur<br/>(lidar_traitement)
    C->>S: Goal {start: true}
    S-->>C: Goal accepté
    loop pour chaque position du servo
        Note over S: bouger le servo,<br/>prendre un scan
        S-->>C: Feedback {progress: 10 %, 20 %, …}
    end
    S-->>C: Result {success, message, point_cloud}
```

Côté serveur, un objectif passe par les états suivants :

```mermaid
stateDiagram-v2
    direction LR
    state "Accepté" as Accepted
    state "En exécution" as Executing
    state "Réussi" as Succeeded
    state "Avorté" as Aborted
    [*] --> Accepted : goal reçu
    Accepted --> Executing : execute_callback démarre
    Executing --> Succeeded : goal_handle.succeed()
    Executing --> Aborted : goal_handle.abort()
    Succeeded --> [*]
    Aborted --> [*]
```

ROS 2 permet aussi au client d'**annuler** une action en cours. Nous ne l'utiliserons pas : par défaut, le serveur refuse les annulations (voir l'encadré plus bas).

### Côté serveur, en code

**1. Créer le serveur**, dans le constructeur du node. On lui donne le type de l'action, son nom et la fonction à exécuter quand un objectif arrive :

```python
from rclpy.action import ActionServer
from jerro_msgs.action import StartScan

self._action_server = ActionServer(self, StartScan, 'perception/start_scan', self.execute_callback)
```

**2. Écrire `execute_callback`**. Elle reçoit un `goal_handle`, qui donne accès à l'objectif (`goal_handle.request`) et permet de répondre au client. Exemple générique, avec un travail fictif :

```python
def execute_callback(self, goal_handle):
    n_etapes = 10
    for i in range(n_etapes):
        time.sleep(0.5)                          # le travail (au Jalon 3 : servo + scan)

        feedback = StartScan.Feedback()
        feedback.progress = 100.0 * (i + 1) / n_etapes
        goal_handle.publish_feedback(feedback)   # donner des nouvelles au client

    goal_handle.succeed()                        # marquer l'objectif comme réussi
    result = StartScan.Result()
    result.success = True
    result.message = "Terminé"
    return result                                # le résultat est envoyé au client
```

À retenir :
- `publish_feedback` peut être appelé autant de fois que l'on veut ;
- on termine **toujours** par `succeed()` (ou `abort()` en cas d'échec) **puis** `return` du résultat.

### Côté client : la ligne de commande

```bash
ros2 action list                                    # actions disponibles
ros2 action info /perception/start_scan             # qui fournit cette action ?
ros2 action send_goal /perception/start_scan jerro_msgs/action/StartScan "{start: true}" --feedback
```

Avec `--feedback`, on voit défiler la progression, puis le résultat :
```
Goal accepted with ID: 3f2a...
Feedback:
    progress: 10.0
Feedback:
    progress: 20.0
...
Result:
    success: true
    message: Terminé
Goal finished with status: SUCCEEDED
```

### ⚠️ Pendant `execute_callback`, le node ne fait rien d'autre

Un node ROS 2 traite ses callbacks **un à la fois**. Tant que `execute_callback` tourne (plusieurs secondes de `time.sleep` et d'attente de scans), aucun autre callback du node n'est exécuté. Trois conséquences pour le Jalon 3 :

1. **On n'attend pas la réponse d'un service** appelé depuis `execute_callback` : la réponse serait traitée par le node… qui est occupé. Le programme attendrait indéfiniment (*interblocage*). On envoie donc la requête avec `call_async` et on continue.
2. **Le listener TF a son propre node et son propre thread** : on lui donne un second node (`self.tf_node`) et `spin_thread=True`, pour que les transformations continuent d'arriver pendant le balayage. Si on lui donnait `self`, ce même node serait occupé par `execute_callback` et TF ne serait plus mis à jour.
3. **Un balayage lancé ne peut pas être annulé** : la demande d'annulation ne serait lue qu'à la fin.

## 2.2.4 Jalon 3 : Implémentation du balayage 3D

Vous allez réécrire le node `lidar_traitement.py` du package `jerro_perception` pour qu'il offre l'action `/perception/start_scan`.

### Vue d'ensemble

```mermaid
flowchart LR
    CLI["ros2 action send_goal"] -- "action<br/>/perception/start_scan" --> LT["lidar_traitement<br/>(votre node)"]
    LT -- "service<br/>/set_servo_pos" --> SV[servomoteur]
    SV -- "/set_joint_states" --> JSP[joint_state_publisher]
    JSP --> RSP[robot_state_publisher]
    RSP -- "TF base_link → lidar" --> LT
    LID[sllidar_ros2] -- "topic /scan" --> LT
    LT -- "topic<br/>/perception/lidar_pc2" --> RV[RViz]
```

### L'outil fourni : la classe ScanAssembler

Le package `laser_assembler` contient la classe `ScanAssembler`. C'est une simple classe Python (aucun node à lancer) : à chaque scan ajouté, elle lit dans TF la transformation `base_link ← lidar` valide à l'instant du scan (votre $^RT_L$ du Jalon 1), transforme les points dans `base_link` et les mémorise.

| Méthode | Rôle |
|---|---|
| `ScanAssembler(tf_buffer, fixed_frame='base_link')` | Crée l'assembleur, qui lira les transformations dans `tf_buffer` |
| `add_scan(scan)` | Ajoute un `LaserScan` (retourne `False` si la TF est introuvable) |
| `to_pointcloud2()` | Retourne le `PointCloud2` assemblé dans `base_link` |
| `clear()` | Vide la mémoire, avant un nouveau balayage |

Le `frame_id` des scans doit être le nom du repère du lidar dans l'URDF (`lidar`) : c'est réglé dans le launch file (`frame_id:=lidar` pour `sllidar_ros2`).

### Prérequis et compilation

Clonez le package `laser_assembler` (et le driver du lidar, si ce n'est pas déjà fait au Lab 2a), puis compilez :

In [ ]:
%%bash
cd ~/ros2_ws/src
git clone https://github.com/Lab-CORO/laser_assembler_ros2
git clone https://github.com/Slamtec/sllidar_ros2.git   # si pas déjà fait au Lab 2a
cd ~/ros2_ws
colcon build --packages-select sllidar_ros2 laser_assembler jerro_perception
source install/setup.bash

### Le squelette à compléter

Remplacez le contenu de `jerro_perception/jerro_perception/lidar_traitement.py` par le squelette ci-dessous. Les parties marquées `TODO` sont à écrire : les étapes suivantes vous guident, une à une.

⚠️ Les constantes `SWEEP_*` limitent la course du servo pour que le lidar ne heurte pas le boîtier du RPI. **Ne les modifiez pas sans l'accord du chargé de laboratoire.**

In [ ]:
import time

import rclpy
from rclpy.node import Node
from rclpy.action import ActionServer
from rclpy.wait_for_message import wait_for_message
import tf2_ros

from sensor_msgs.msg import LaserScan, PointCloud2
from jerro_msgs.action import StartScan
from jerro_msgs.srv import SetServoPos

from laser_assembler.scan_assembler import ScanAssembler


class LidarTraitement(Node):
    # Positions du servo, en largeur d'impulsion (µs). NE PAS MODIFIER sans le chargé de lab.
    SWEEP_MIN_PULSE = 1300
    SWEEP_MAX_PULSE = 1700
    SWEEP_STEP_PULSE = 20
    CENTER_PULSE = 1500
    SETTLE_TIME = 0.3   # s : temps laissé au servo pour atteindre sa position

    def __init__(self):
        super().__init__('lidar_traitement')

        # Nuage 3D publié à la fin de chaque balayage
        self.pc2_pub = self.create_publisher(PointCloud2, 'perception/lidar_pc2', 10)

        # TF : le listener a son propre node et son propre thread, pour rester à jour
        # pendant execute_callback (voir l'encadré du Jalon 2)
        self.tf_buffer = tf2_ros.Buffer()
        self.tf_node = rclpy.create_node('lidar_traitement_tf')
        self.tf_listener = tf2_ros.TransformListener(self.tf_buffer, self.tf_node, spin_thread=True)
        self.assembler = ScanAssembler(self.tf_buffer, fixed_frame='base_link')

        # TODO 1 : créer le client du service 'set_servo_pos' (type SetServoPos)
        #          et attendre qu'il soit disponible.

        # TODO 3 : créer le serveur d'action 'perception/start_scan' (type StartScan),
        #          qui appelle self.execute_callback.

        self.get_logger().info("lidar_traitement prêt.")

    def move_servo(self, pulse_us):
        """Envoie le servo à la position pulse_us (en µs) et attend qu'il y soit."""
        # TODO 2
        pass

    def execute_callback(self, goal_handle):
        """Réalise un balayage 3D complet."""
        # TODO 4 : balayer de SWEEP_MIN_PULSE à SWEEP_MAX_PULSE et assembler les scans.

        # TODO 5 : publier le nuage, remettre le servo au centre et retourner le résultat.
        pass


def main(args=None):
    rclpy.init(args=args)
    node = LidarTraitement()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.tf_node.destroy_node()
        node.destroy_node()
        rclpy.shutdown()


if __name__ == '__main__':
    main()

### Étape 1 : commander le servo (TODO 1 et 2)

**Dans le robot :** le node `servomoteur` (package `jerro_drivers`) pilote le PWM du RPI et offre le service `/set_servo_pos`. Votre node n'a qu'à lui envoyer une requête :

```
int16 position   # largeur d'impulsion en µs (1000 à 2000, 1500 = centre)
---
bool result
```

**À faire :**
- **TODO 1** : dans le constructeur, créez un client pour ce service et attendez qu'il soit disponible.
- **TODO 2** : dans `move_servo`, envoyez la requête **sans attendre la réponse** (voir l'encadré du Jalon 2), puis laissez au servo le temps `SETTLE_TIME` d'atteindre sa position.

**Vérifier :** testez d'abord le service à la main (`ros2 service call /set_servo_pos jerro_msgs/srv/SetServoPos "{position: 1400}"`). Puis appelez temporairement `self.move_servo(1400)` à la fin du constructeur : dans RViz, le repère `lidar` doit pivoter. Retirez ensuite cet appel.

### Étape 2 : un serveur d'action « à vide » (TODO 3)

**Dans le robot :** avant de faire bouger le matériel, on vérifie que l'action fonctionne.

**À faire :**
- **TODO 3** : créez le serveur d'action dans le constructeur.
- Écrivez provisoirement un `execute_callback` qui reprend l'exemple générique du Jalon 2 : une boucle avec `time.sleep`, un feedback par tour, puis `succeed()` et le résultat.

**Vérifier :**
```bash
ros2 action send_goal /perception/start_scan jerro_msgs/action/StartScan "{start: true}" --feedback
```
La progression doit défiler jusqu'à 100 %, puis `Goal finished with status: SUCCEEDED`.

### Étape 3 : le vrai balayage (TODO 4)

**Dans le robot :** on remplace le travail fictif par le balayage. Pour chaque position du servo, on prend **un seul** scan, au moment où le lidar est immobile : c'est le rôle de `wait_for_message` (un subscriber, lui, recevrait les scans en continu, même pendant que le servo bouge).

```mermaid
flowchart TD
    A(["Goal reçu"]) --> B["assembler.clear()"]
    B --> C["move_servo : position suivante"]
    C --> D["wait_for_message : un scan sur /scan"]
    D --> E["assembler.add_scan(scan)"]
    E --> F["publish_feedback"]
    F --> G{"Dernière position ?"}
    G -- non --> C
    G -- oui --> H["TODO 5 : fin du balayage"]
```

**À faire (TODO 4) :**
- videz l'assembleur au début du balayage ;
- parcourez les positions de `SWEEP_MIN_PULSE` à `SWEEP_MAX_PULSE` par pas de `SWEEP_STEP_PULSE` (indice : `range`) ;
- à chaque position : bougez le servo, attendez un scan, ajoutez-le à l'assembleur, publiez la progression (en %).

Le premier déplacement (du centre jusqu'à `SWEEP_MIN_PULSE`) est plus long que les autres : laissez une seconde de plus au servo.

**Vérifier :** affichez dans le log le nombre de scans assemblés (`self.assembler.n_scans`) à la fin de la boucle. Il doit valoir le nombre de positions.

### Étape 4 : le résultat (TODO 5)

**À faire :**
- récupérez le nuage avec `to_pointcloud2()` et publiez-le sur `perception/lidar_pc2` ;
- remettez le servo au centre (`CENTER_PULSE`) ;
- terminez l'action (`succeed()`) et retournez le résultat (`success` et `message`).

Ne mettez **pas** le nuage dans le champ `point_cloud` du résultat : `ros2 action send_goal` afficherait ses ~66 000 points dans le terminal, ce qui est très lent sur le RPI. Le nuage se regarde dans RViz, sur `/perception/lidar_pc2`. Le champ `point_cloud` servira au laboratoire 2c.

### Fiche d'aide

| Besoin | API |
|---|---|
| Client de service | `self.create_client(Type, 'nom')`, `client.wait_for_service()`, `client.call_async(requete)` |
| Créer une requête | `req = SetServoPos.Request()` puis `req.position = ...` |
| Serveur d'action | `ActionServer(self, Type, 'nom', self.execute_callback)` |
| Dans `execute_callback` | `goal_handle.publish_feedback(fb)`, `goal_handle.succeed()`, `return resultat` |
| Messages de l'action | `StartScan.Feedback()`, `StartScan.Result()` |
| Recevoir un seul message | `ok, msg = wait_for_message(LaserScan, self, '/scan')` |
| Assembler | `self.assembler.clear()`, `.add_scan(scan)`, `.to_pointcloud2()`, `.n_scans` |
| Publier | `self.pc2_pub.publish(msg)` |

### Test final avec RViz

⚠️ **Avant de tester votre code, montrez-le au chargé de laboratoire pour autorisation !** Des erreurs dans les commandes du servomoteur peuvent causer des bris matériels.

1. Connectez-vous au RPI par VNC
2. Démarrez le daemon pigpio (nécessaire au node `servomoteur`) :

In [ ]:
%%bash
sudo pigpiod

3. Lancez le launch file. Il démarre la description du robot (TF), le servomoteur, le lidar, la caméra et vos deux nodes (`cam_traitement` et `lidar_traitement`) :

> Dans le launch file, ne donnez **pas** de `name=` au node `lidar_traitement` : ce paramètre renommerait tous les nodes du processus, y compris `lidar_traitement_tf`.

In [ ]:
%%bash
ros2 launch jerro_perception lidar_et_cam.launch.py

4. Lancez rviz :

In [ ]:
%%bash
ros2 run rviz2 rviz2

5. **Configuration dans RViz :**
   - « Fixed Frame » : `base_link`
   - Ajoutez « TF », « LaserScan » et « PointCloud2 »
   - Pour TF : sélectionnez `base_link`, `servomotor` et `lidar`
   - Pour LaserScan : topic `/scan`
   - Pour PointCloud2 : topic `/perception/lidar_pc2`

6. **Démarrer l'acquisition** :

In [ ]:
%%bash
ros2 action send_goal /perception/start_scan jerro_msgs/action/StartScan "{start: true}" --feedback

### Résultat attendu

Vous devriez observer :
- La rotation du repère `lidar` par rapport à `base_link`
- Les données 2D du lidar au fur et à mesure
- Le nuage de points 3D complet une fois le balayage terminé

![Figure 2.9 - Résultat RViz](figure_2_9.png)
*Figure 2.9: Aperçu du type de résultat attendu, dans l'application « rviz »*

## 2.2.5 Démonstration et questions

### Démonstration au chargé de laboratoire

La démonstration et les questions supplémentaires seront complétées à la fin de la **prochaine séance (laboratoire #2c)**.

### Points à vérifier

- [ ] Transformation homogène $^RT_L$ développée et comparée avec la TF `base_link → lidar`
- [ ] Commande du servo par le service `/set_servo_pos`
- [ ] Serveur d'action `/perception/start_scan` fonctionnel, avec feedback
- [ ] Nuage de points 3D visible dans RViz

### Pour aller plus loin

- Avec l'accord du chargé de laboratoire, modifiez `SWEEP_STEP_PULSE` et `SETTLE_TIME`
- Observez l'impact sur la densité et la qualité du nuage de points, ainsi que sur la durée du balayage